# Ensemble Predictions

## Overview

This notebook generates ensemble predictions by combining:
1. T5-small model predictions
2. mT5-small model predictions
3. Retrieval-based (TF-IDF) predictions

Ensemble methods improve robustness by averaging or voting across different approaches.

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, MT5Tokenizer, MT5ForConditionalGeneration
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load test data
test_df = pd.read_csv('test.csv')
train_df = pd.read_csv('train_augmented.csv')

print(f"Test size: {len(test_df)}")
print(f"Train size: {len(train_df)}")

## 1. Load Both Models (T5 and mT5)

In [ ]:
models = {}
tokenizers = {}

# Load T5-small
print("Loading T5-small model...")
try:
    tokenizers['t5'] = T5Tokenizer.from_pretrained("./ensemble_models/t5_small")
    models['t5'] = T5ForConditionalGeneration.from_pretrained("./ensemble_models/t5_small")
    models['t5'] = models['t5'].to(DEVICE)
    print("Loaded T5-small from ensemble_models.")
except:
    print("T5-small not found in ensemble_models, using base model...")
    tokenizers['t5'] = T5Tokenizer.from_pretrained("t5-small")
    models['t5'] = T5ForConditionalGeneration.from_pretrained("t5-small")
    models['t5'] = models['t5'].to(DEVICE)

# Load mT5-small
print("\nLoading mT5-small model...")
try:
    tokenizers['mt5'] = MT5Tokenizer.from_pretrained("./ensemble_models/mt5_small")
    models['mt5'] = MT5ForConditionalGeneration.from_pretrained("./ensemble_models/mt5_small")
    models['mt5'] = models['mt5'].to(DEVICE)
    print("Loaded mT5-small from ensemble_models.")
except Exception as e:
    print(f"mT5-small not found: {e}")
    print("Will use only T5 model.")
    models['mt5'] = None

print(f"\nModels loaded: {list(models.keys())}")

## 2. Generate Neural Model Predictions

In [ ]:
def generate_predictions(model, tokenizer, texts, device):
    """Generate predictions from a model."""
    test_inputs = tokenizer(
    texts,
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'
    )
    test_inputs = {k: v.to(device) for k, v in test_inputs.items()}

    outputs = model.generate(
        input_ids=test_inputs['input_ids'],
        attention_mask=test_inputs['attention_mask'],
        max_length=128,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

test_texts = ["translate Akkadian to English: " + str(t) for t in test_df['transliteration'].tolist()]

# Generate predictions from each model
all_neural_preds = {}

for model_name, model in models.items():
    if model is not None:
        print(f"\nGenerating predictions from {model_name}...")
        all_neural_preds[model_name] = generate_predictions(
            model, tokenizers[model_name], test_texts, DEVICE
        )
        print(f"Sample prediction: {all_neural_preds[model_name][0][:80]}...")

## 3. Combine Neural Model Predictions

In [ ]:
# Combine predictions from both models using simple averaging/selection
print("\nCombining predictions from both neural models...")

if len(all_neural_preds) > 1:
    # Average: Take the longer prediction (usually more informative)
    combined_neural_preds = []
    for i in range(len(test_df)):
        preds = [all_neural_preds[model][i] for model in all_neural_preds]
        # Select the longest prediction (tends to be more complete)
        combined_neural_preds.append(max(preds, key=len))
    print("Using combined predictions from T5 + mT5")
else:
    combined_neural_preds = list(all_neural_preds.values())[0]
    print(f"Using single model: {list(all_neural_preds.keys())[0]}")

print("\nCombined neural predictions:")
for i, pred in enumerate(combined_neural_preds[:3]):
    print(f"  {i}: {pred[:80]}...")

## 4. Generate Retrieval Predictions

In [ ]:
print("\nGenerating retrieval predictions...")

train_df = train_df.dropna(subset=['transliteration', 'translation'])
train_texts = train_df['transliteration'].tolist()
train_labels = train_df['translation'].tolist()
test_texts_raw = test_df['transliteration'].tolist()

# TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 3),
    analyzer='char_wb',
    sublinear_tf=True
)

train_vectors = vectorizer.fit_transform(train_texts)
test_vectors = vectorizer.transform(test_texts_raw)

similarities = cosine_similarity(test_vectors, train_vectors)

# Get retrieval predictions
retrieval_preds = []
for i in range(len(test_texts_raw)):
    best_idx = np.argmax(similarities[i])
    retrieval_preds.append(train_labels[best_idx])

print("Retrieval predictions:")
for i, pred in enumerate(retrieval_preds[:3]):
    print(f"  {i}: {pred[:80]}...")
    print(f"     Similarity: {similarities[i][best_idx]:.3f}")

## 5. Ensemble: Combine Neural and Retrieval Predictions

In [ ]:
print("\n--- Creating Final Ensemble ---")

# Method: Use retrieval for high similarity, neural for low similarity
ensemble_preds = []
for i in range(len(test_texts_raw)):
    max_sim = similarities[i][np.argmax(similarities[i])]
    
    # If retrieval similarity is high, prefer retrieval
    if max_sim > 0.6:
        ensemble_preds.append(retrieval_preds[i])
    else:
        # Otherwise use combined neural prediction (from both models)
        ensemble_preds.append(combined_neural_preds[i])

print("Ensemble predictions:")
for i, pred in enumerate(ensemble_preds[:5]):
    max_sim = similarities[i][np.argmax(similarities[i])]
    source = "retrieval" if max_sim > 0.6 else "neural_ensemble"
    print(f"  {i} [{source}]: {pred[:80]}...")

## 6. Save Submission

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': ensemble_preds
})

# Save submission
submission.to_csv('submission_ensemble.csv', index=False)
print("Ensemble submission saved to submission_ensemble.csv")

print("\n--- Final Submission ---")
print(submission)

## 7. Analysis

In [ ]:
# Also save detailed predictions for analysis
all_preds = pd.DataFrame({
    'id': test_df['id'],
    'transliteration': test_df['transliteration'],
    't5_pred': all_neural_preds.get('t5', [''] * len(test_df)),
    'mt5_pred': all_neural_preds.get('mt5', [''] * len(test_df)),
    'combined_neural': combined_neural_preds,
    'retrieval_pred': retrieval_preds,
    'ensemble_pred': ensemble_preds,
    'retrieval_similarity': [similarities[i][np.argmax(similarities[i])] for i in range(len(test_df))]
})

all_preds.to_csv('all_predictions.csv', index=False)
print("All predictions saved to all_predictions.csv")

# Count predictions by source
sources = []
for i in range(len(test_texts_raw)):
    max_sim = similarities[i][np.argmax(similarities[i])]
    sources.append("retrieval" if max_sim > 0.6 else "neural_ensemble")

from collections import Counter
source_counts = Counter(sources)
print(f"\n--- Ensemble Analysis ---")
print(f"Predictions from neural ensemble: {source_counts.get('neural_ensemble', 0)}")
print(f"Predictions from retrieval: {source_counts.get('retrieval', 0)}")

# Show comparison
print("\n--- Prediction Comparison ---")
for i in range(min(3, len(test_df))):
    print(f"\nTest {i}:")
    t5_p = all_neural_preds.get('t5', [''] * len(test_df))
    mt5_p = all_neural_preds.get('mt5', [''] * len(test_df))
    t5_pred = t5_p[i] if i < len(t5_p) else 'N/A'
    mt5_pred = mt5_p[i] if i < len(mt5_p) else 'N/A'
    print(f"  T5:     {t5_pred[:60]}...")
    print(f"  mT5:    {mt5_pred[:60]}...")
    print(f"  Ensemble: {ensemble_preds[i][:60]}...")